### Ситников Сергей Олегович М3239
# **Лабораторная 1**

## Задача 1 (Вариант 1)
В городе с населением $n+1$ человек некто узнает новость. Он передает ее первому встречному, тот еще одному и т.д. На каждом шаге услышавший новость может сообщить её любому из $n$ человек с одинаковыми вероятностями. Найти вероятность того, что в течение $r$ единиц времени новость никто не услышит дважды. Решить ту же задачу в предположении, что на каждом шаге услышавший на предыдущей итерации сообщение передает его группе из $m$ человек.

В первой версии на $j$-том шаге у нас $j$ человек осведомлены о новости, остаётся $n+1-j$ людей, которые неосведомлены. Чтобы на $r$-ом шаге никто не узнал новость дважды, эта новость должна быть произнесена за всё время только тем, кто о ней ещё не был осведомлен. Итого формула выглядит как 
$$ P(r) = \frac{n}{n} * \frac{n - 1}{n} * \frac{n - 2}{n} * \cdots * \frac{n - r + 1}{n} = \prod_{k=1}^{r} \frac{n-k+1}{n}=\frac{n!}{(n-r)!n^r},\quad 0 \le r \le n$$
При $r>n$ вероятность равна нулю.

In [1]:
import random
import math
random.seed("Sitnikov Sergey")

def simulate_one(n: int, r: int) -> bool:
    knows = [False] * (n + 1)
    knows[0] = True
    current_teller = 0
    for _ in range(r):
        index_chosen = random.randint(0, n - 1)
        if index_chosen >= current_teller:
            index_chosen += 1
        if knows[index_chosen]:
            return False
        knows[index_chosen] = True
        current_teller = index_chosen
    return True

COUNT_EXPERIMENTS = 200_000
print("n     r      simulation    formula       delta to formula")
for n in range(4, 10):
    for r in range(1, n + 1):
        good_count = 0
        for _ in range(COUNT_EXPERIMENTS):
            good_count += simulate_one(n, r)
        simulation = good_count / COUNT_EXPERIMENTS
        formula = math.factorial(n) / math.factorial(n - r) / (n**r)
        delta = (simulation - formula) / formula * 100
        print(f"{n:<3} {r:>3} {simulation:>13.6f} {formula:>13.6f} {delta:+10.2f}%")

n     r      simulation    formula       delta to formula
4     1      1.000000      1.000000      +0.00%
4     2      0.751525      0.750000      +0.20%
4     3      0.374110      0.375000      -0.24%
4     4      0.094495      0.093750      +0.79%
5     1      1.000000      1.000000      +0.00%
5     2      0.800570      0.800000      +0.07%
5     3      0.481250      0.480000      +0.26%
5     4      0.191185      0.192000      -0.42%
5     5      0.038530      0.038400      +0.34%
6     1      1.000000      1.000000      +0.00%
6     2      0.832680      0.833333      -0.08%
6     3      0.556110      0.555556      +0.10%
6     4      0.277125      0.277778      -0.24%
6     5      0.092230      0.092593      -0.39%
6     6      0.015110      0.015432      -2.09%
7     1      1.000000      1.000000      +0.00%
7     2      0.856665      0.857143      -0.06%
7     3      0.612035      0.612245      -0.03%
7     4      0.351735      0.349854      +0.54%
7     5      0.149465      0.1

Теперь вторая версия задачи, теперь сообщается новость не одному человеку за раз, а сразу m людям. Тогда на $j$-ом шагу нужно выбрать m людей только среди тех, кто еще не осведомлён. Неосведомлённых людей $n - (j-1)m$ (на каждом шагу осведомляется на m людей больше, это всё при учёте, что мы не хотим, чтобы одному и тому же человеку дважды была рассказана новость). Передатчик на $j$-ом шаге выбирается среди тех, кому рассказали новость на $(j-1)$-ом шаге. Значит формула будет выглядеть так:
$$ P(r) = \prod_{j=1}^r \frac{C_{n-(j-1)m}^m}{C_n^m}, \quad mr \le n$$
При $mr > n$ вероятность равна нулю.

In [43]:
import random
import math
random.seed("Sitnikov Sergey")

def simulate_one(n: int, m: int, r: int) -> bool:
    knows = [False] * (n + 1)
    knows[0] = True
    current_teller = 0
    for _ in range(r):
        possible_receivers = [
            person for person in range(n + 1)
            if person != current_teller
        ]
        group = random.sample(possible_receivers, m)
        for person in group:
            if knows[person]:
                return False
            knows[person] = True
        current_teller = random.choice(group)
    return True


COUNT_EXPERIMENTS = 200_000
print("n     m   r      simulation    formula       delta to formula")

for n in range(4, 10):
    for m in range(1, n + 1):
        for r in range(1, n // m + 1):
            good_count = 0
            for _ in range(COUNT_EXPERIMENTS):
                good_count += simulate_one(n, m, r)
            simulation = good_count / COUNT_EXPERIMENTS
            formula = math.prod(
                math.comb(n - (j - 1) * m, m) / math.comb(n, m)
                for j in range(1, r + 1)
            )
            delta = (simulation - formula) / formula * 100
            print(
                f"{n:<3} {m:>3} {r:>3} "
                f"{simulation:>13.6f} {formula:>13.6f} "
                f"{delta:+10.2f}%"
            )

n     m   r      simulation    formula       delta to formula
4     1   1      1.000000      1.000000      +0.00%
4     1   2      0.750490      0.750000      +0.07%
4     1   3      0.375810      0.375000      +0.22%
4     1   4      0.093080      0.093750      -0.71%
4     2   1      1.000000      1.000000      +0.00%
4     2   2      0.166465      0.166667      -0.12%
4     3   1      1.000000      1.000000      +0.00%
4     4   1      1.000000      1.000000      +0.00%
5     1   1      1.000000      1.000000      +0.00%
5     1   2      0.799920      0.800000      -0.01%
5     1   3      0.479905      0.480000      -0.02%
5     1   4      0.191495      0.192000      -0.26%
5     1   5      0.038300      0.038400      -0.26%
5     2   1      1.000000      1.000000      +0.00%
5     2   2      0.297990      0.300000      -0.67%
5     3   1      1.000000      1.000000      +0.00%
5     4   1      1.000000      1.000000      +0.00%
5     5   1      1.000000      1.000000      +0.00%
6 

## Задача 2 (Вариант 2)
Случайная точка $A$ имеет равномерное распределение в правильном
$n$-угольнике. Найти вероятность $P_n$, что точка $A$ находится
ближе к границе многоугольника, чем к его диагоналям.
Найти числа $C$, $a$, что
$$P_n = Cn^a(1 + o(1)), \qquad n \to \infty.$$

![second](img/second.png)

Рассмотрим правильный $n$-угольник. Пусть длина стороны будет $s$. Занумеруем вершины: $v_0$, $v_1$, $\cdots$, $v_{n-1}$. Рассмотрим $\triangle v_0v_1O$, где $O$ - середина многоугольника. Пусть точка попала в него. Тогда когда расстояние от неё до стороны $v_0v_1$ ближе, чем до диагоналей? Необходимое (но не достаточное) условие этому: точка должна лежать в треугольнике, образованном пересечением диагоналей $v_0v_2$ и $v_1v_{n-1}$. Иначе бы путь до стороны $v_0v_1$ состоял бы из пересечения с какой-либо из этих диагоналей, а значит расстояние до стороны $v_0v_1$ явно не ближе, чем до диагоналей. На рисунке пересечение эти двух диагоналей я обозначил буквой $A$. Давайте найдём угол $\angle v_1v_0v_2$, обозначенный на рисунке как $\alpha$. Рассмотрим для этого $\triangle v_0v_1v_2$. $\angle v_1 = \frac{(n-2)\pi}{n}$ по известной со школы формуле углов правильного многоугольника. Так как этот треугольник равнобедренный, оставшиеся два угла равны, значит $$\angle v_0 = \angle v_2 = \frac{\pi - \frac{(n-2)\pi}{n}}{2} = \frac{\pi}{n} = \alpha$$. Знаем, что вершины, равноудаленные от $v_0v_1$ и $v_0A$, лежат на биссектрисе угла, образованного этими сторонами. Обозначим пересечение биссектрис из $\angle Av_1v_0$ и $\angle v_1v_0A$ как $B$. Получаем, что все вершины, лежащие в $\triangle v_0v_1B$ находятся ближе к стороне $v_0v_1$, чем к диагоналям многоугольника. Осталось найти его площадь. Опустим из $B$ высоту на сторону многоугольник, обозначив точку пересечения как $v_{0.5}$. Обозначим длину этой высоты как $h$. Тогда: 
$$\frac{h}{(\frac{s}{2})}=\tg \frac{\pi}{2n}$$
$$2h=s\tg \frac{\pi}{2n}$$
$$h=\frac{s\tg \frac{\pi}{2n}}{2}$$
$$S_{\triangle v_0v_{0.5}B}=\frac{\frac{s\tg \frac{\pi}{2n}}{2} \cdot \frac{s}{2}}{2}$$
$$S_{\triangle v_0v_1B}=\frac{s\tg \frac{\pi}{2n}}{2} \cdot \frac{s}{2}=\frac{s^2}{4} \tg \frac{\pi}{2n}$$
Площадь правильного многоугольника обозначим за $$S_n = \frac{n}{4} s^2 \ctg \frac{\pi}{n}$$
Тогда получаем $$P_n = \frac{nS_{\triangle v_0v_1B}}{S_n}= \frac{n\frac{s^2}{4} \tg \frac{\pi}{2n}}{\frac{n}{4} s^2 \ctg \frac{\pi}{n}}=\tg \frac{\pi}{2n} \tg \frac{\pi}{n}=\frac{1-\cos{\frac{\pi}{n}}}{\cos \frac{\pi}{n}} = \frac{1}{\cos{\frac{\pi}{n}}}- 1$$
Теперь перейдём к асимптотике. Так как у нас по условию $n \to \infty$, то $\frac{\pi}{n} \to 0$. Тогда
$$P_n =\tg \frac{\pi}{2n} \tg \frac{\pi}{n} \sim \frac{\pi}{2n} \cdot \frac{\pi}{n} = \frac{\pi^2}{2}n^{-2}$$
$$P_n = \frac{\pi^2}{2}n^{-2}(1+o(1))$$
Значит 
$$C=\frac{\pi^2}{2}, a=-2$$
Перейдём к симуляции. Чтобы выбрать равномерно точку в правильном $n$-угольнике, достаточно выбирать точку в квадрате, куда вписан наш $n$-угольник, а если точка в него не попала, заново выбирать. Так делать, пока точка не попадёт в $n$-угольник. Действительно, никакая область этого $n$-угольника не получает преимуществ над другими из-за такого алгоритма, а значит мы равномерно выбираем точку внутри $n$-угольника.

In [6]:
import random
import math
random.seed("Sitnikov Sergey")

def create_poly(n):
    vertices = [
        (
            math.cos(2 * math.pi * k / n),
            math.sin(2 * math.pi * k / n)
        )
        for k in range(n)
    ]
    sides = [
        (vertices[i], vertices[(i + 1) % n])
        for i in range(n)
    ]
    diagonals = [
        (vertices[i], vertices[j])
        for i in range(n)
        for j in range(i + 1, n)
        if j - i != 1 and (i, j) != (0, n - 1)
    ]
    return sides, diagonals

def check_inside(point, sides):
    x, y = point
    for (ax, ay), (bx, by) in sides:
        cross = (bx - ax) * (y - ay) - (by - ay) * (x - ax)
        if cross < 0: return False
    return True

def from_point_to_segment(A, C, D):
    CD = (D[0] - C[0], D[1] - C[1])
    C__A = (A[0] - C[0], A[1] - C[1])
    DC = (C[0] - D[0], C[1] - D[1])
    A__D = (A[0] - D[0], A[1] - D[1])
    right_poluploskost = CD[0] * C__A[0] + CD[1] * C__A[1] >= 0
    left_poluploskost = DC[0] * A__D[0] + DC[1] * A__D[1] >= 0
    if left_poluploskost and right_poluploskost:
        cross = CD[0] * C__A[1] - CD[1] * C__A[0]
        length = math.sqrt(CD[0]**2 + CD[1]**2)
        return abs(cross) / length
    return min(
        math.sqrt(C__A[0]**2 + C__A[1]**2),
        math.sqrt(A__D[0]**2 + A__D[1]**2)
    )

def simulate_one(sides, diagonals):
    while True:
        point = (
            random.uniform(-1, 1),
            random.uniform(-1, 1)
        )
        if check_inside(point, sides):
            break
    distance_boundary = min(
        from_point_to_segment(point, start, end)
        for start, end in sides
    )
    distance_diagonal = min(
        from_point_to_segment(point, start, end)
        for start, end in diagonals
    )
    return distance_boundary < distance_diagonal

COUNT_EXPERIMENTS = 200_000

print("n        simulation    formula       delta to formula")

for n in range(4, 15):
    sides, diagonals = create_poly(n)
    good_count = 0
    for _ in range(COUNT_EXPERIMENTS):
        good_count += simulate_one(sides, diagonals)
    simulation = good_count / COUNT_EXPERIMENTS
    formula = math.tan(math.pi / n) * math.tan(math.pi / (2 * n))
    delta = (simulation - formula) / formula * 100
    print(
        f"{n:<3} {simulation:>13.6f} "
        f"{formula:>13.6f} {delta:+10.2f}%"
    )

n        simulation    formula       delta to formula
4        0.413125      0.414214      -0.26%
5        0.237110      0.236068      +0.44%
6        0.154610      0.154701      -0.06%
7        0.109020      0.109916      -0.82%
8        0.082655      0.082392      +0.32%
9        0.063665      0.064178      -0.80%
10       0.050145      0.051462      -2.56%
11       0.043130      0.042217      +2.16%
12       0.035820      0.035276      +1.54%
13       0.030010      0.029928      +0.27%
14       0.025380      0.025717      -1.31%


## Задача 3 (Вариант 1)
Пусть имеются две независимые серии испытаний Бернулли на $n$ опытов
в каждой с вероятностью успеха $p$, $S_i$ — количество успехов
в $n$ испытаниях в $i$-й серии. Найти вероятность
$$P(S_1 = k \mid S_1 + S_2 = m).$$
По формуле условной вероятности:
$$P(S_1 = k \mid S_1 + S_2 = m) = \frac{P(S_1 = k \; \& \; S_1 + S_2 = m)}{P(S_1 + S_2 = m)}$$
$$P(S_1 = k \; \& \; S_1 + S_2 = m) = P(S_1 = k \; \& \; S_2 = m - k)$$
Т.к. $S_1,S_2$ независимы, то $$P(S_1 = k \; \& \; S_1 + S_2 = m) = P(S_1 = k \; \& \; S_2 = m - k) = P(S_1=k)P(S_2=m-k)$$
Две независимве серии испытаний Бернулли на $n$ опытов - это всё равно, что одна серия испытания на $2n$ опытов.

$$P(S_1 = k \mid S_1 + S_2 = m) = \frac{P(S_1 = k \; \& \; S_1 + S_2 = m)}{P(S_1 + S_2 = m)}=\frac{P(S_1=k)P(S_2=m-k)}{P(S_1 + S_2 = m)}$$
$$=\frac{C_n^kp^kq^{n-k}C_n^{m-k}p^{m-k}q^{n-m+k}}{C_{2n}^mp^mq^{2n-m}}=\frac{C_n^kC_n^{m-k}}{C_{2n}^m}$$
С условиями: $$0 \le k \le n, \; 0 \le m-k \le n \Rightarrow \max(0,m-n) \le k \le \min(n,m)$$ 

## Задача 4 (Вариант 3)
Биномиальное распределение и нормальное (интервалы)

Из замечания к интегральной предельной теореме Муавра-Лапласа имеем, что для $-\infty \le A < B \le +\infty$ $$P(A < S_n \le B) - \left( \Phi\left(\frac{B-np}{\sqrt{npq}}\right) - \Phi\left(\frac{A-np}{\sqrt{npq}}\right) \right) \longrightarrow 0, \qquad n \to \infty.$$ 

In [ ]:
import random
from statistics import NormalDist

random.seed("Sitnikov Sergey")

def simulate_one(n, p):
    successes = 0
    for _ in range(n):
        if random.random() < p:
            successes += 1
    return successes


def get(A, B, n, p): 
    return NormalDist().cdf((B-n*p)/(n*p*(1-p))**0.5) \
        - NormalDist().cdf((A-n*p)/(n*p*(1-p))**0.5)


COUNT_EXPERIMENTS = 20_000

TESTS = {
    0.1: {
        100: [(7, 13),    (4, 16),    (10, 16)],
        300: [(25, 35),   (20, 40),   (30, 40)],
        500: [(44, 56),   (37, 63),   (50, 63)],
        1000:  [(91, 109),    (82, 118),    (100, 118)],
        3000:  [(284, 316),   (268, 332),   (300, 332)],
        10000: [(970, 1030),  (940, 1060),  (1000, 1060)],
    },
    0.5: {
        100: [(45, 55),   (40, 60),   (50, 60)],
        300: [(142, 158), (133, 167), (150, 167)],
        500: [(239, 261), (228, 272), (250, 272)],
        1000:  [(485, 515),   (469, 531),   (500, 531)],
        3000:  [(1473, 1527), (1446, 1554), (1500, 1554)],
        10000: [(4950, 5050), (4900, 5100), (5000, 5100)],
    },
    0.9: {
        100: [(87, 93),   (84, 96),   (90, 96)],
        300: [(265, 275), (260, 280), (270, 280)],
        500: [(444, 456), (437, 463), (450, 463)],
        1000:  [(891, 909),   (882, 918),   (900, 918)],
        3000:  [(2684, 2716), (2668, 2732), (2700, 2732)],
        10000: [(8970, 9030), (8940, 9060), (9000, 9060)],
    },
}

print(
    "n     p   L    R         simulation    approx        delta"
)

for p in TESTS.keys():
    for n in TESTS[p].keys():
        for (L, R) in TESTS[p][n]:
            good_count = sum(L <= simulate_one(n, p) <= R for _ in range(COUNT_EXPERIMENTS))
            simulation = good_count / COUNT_EXPERIMENTS
            approx = get(L - 0.00001, R, n, p)
            delta = (simulation - approx) / simulation * 100
            print(
                    f"{n:<5} {p:<3} {L:<4} {R:<4} {simulation:>13.6f} "
                    f"{approx:>13.6f} {delta:+10.2f}%"
                )


n     p   L    R         simulation    approx        delta
100   0.1 7    13        0.757750      0.682690      +9.91%
100   0.1 4    16        0.973900      0.954500      +1.99%
100   0.1 10   16        0.527900      0.477251      +9.59%
300   0.1 25   35        0.708200      0.664077      +6.23%
300   0.1 20   40        0.956200      0.945708      +1.10%
300   0.1 30   40        0.501800      0.472855      +5.77%
500   0.1 44   56        0.662400      0.628907      +5.06%
500   0.1 37   63        0.958700      0.947368      +1.18%
500   0.1 50   63        0.498000      0.473684      +4.88%
1000  0.1 91   109       0.688400      0.657219      +4.53%
1000  0.1 82   118       0.948100      0.942220      +0.62%
1000  0.1 100  118       0.491800      0.471111      +4.21%
3000  0.1 284  316       0.687450      0.669809      +2.57%
3000  0.1 268  332       0.952750      0.948520      +0.44%
3000  0.1 300  332       0.488500      0.474260      +2.91%
10000 0.1 970  1030      0.688800      0.